In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
with open('/content/drive/MyDrive/DA/Страхування телефонів/mvswantedmt.json', 'r', encoding='utf-8-sig') as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [ ]:
df.head()

,ID,OVD,INSERT_DATE,NZ,IMEI,NK,DK,DTL
0,3020311526666265,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06T14:49:38,SAMSUNG,356631100673435,54825,2020-11-05T00:00:00,None
1,3020311527577481,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06T14:49:38,SAMSUNG,357665057329129,63543,2020-10-23T00:00:00,None
2,3020311527617564,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06T14:49:38,XIAOMI,867397033107006,54835,2020-11-05T00:00:00,None
3,3020311527958136,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06T14:49:39,XIAOMI,867397033107014,54835,2020-11-05T00:00:00,None
4,3020311528238684,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06T14:49:39,SAMSUNG,357665057329137,63543,2020-10-23T00:00:00,None


# Блок 1
##Очищення даних


In [ ]:
print(df.shape)

(1049037, 8)


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1049037 entries, 0 to 1049036
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   ID           1049037 non-null  object
 1   OVD          1049037 non-null  object
 2   INSERT_DATE  1046489 non-null  object
 3   NZ           1048944 non-null  object
 4   IMEI         1049037 non-null  object
 5   NK           964804 non-null   object
 6   DK           964891 non-null   object
 7   DTL          240418 non-null   object
dtypes: object(8)
memory usage: 64.0+ MB


Дивлюся кількість дублікатів та видаляю

In [ ]:
df.duplicated().sum()

np.int64(225)

In [ ]:
df=df.drop_duplicates()

Цікаво якщо подивитись кількість дублікатів по IMEI або ID кількість буде значно більше ніж загальна кількість дублікатів, що свідчить що в даному датасеті ні ID , ні IMEI не можна вважати унікальними ключами

In [ ]:
df.duplicated(subset=['ID']).sum()

np.int64(7416)

In [ ]:
df.duplicated(subset=['IMEI']).sum()

np.int64(12908)

Записи з пропущеними записами дати не підходять для подальшого аналізу , тож видаляю їх

In [ ]:
df=df.dropna(subset=['DK', 'INSERT_DATE'])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 964667 entries, 0 to 1049036
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   ID           964667 non-null  object
 1   OVD          964667 non-null  object
 2   INSERT_DATE  964667 non-null  object
 3   NZ           964574 non-null  object
 4   IMEI         964667 non-null  object
 5   NK           964537 non-null  object
 6   DK           964667 non-null  object
 7   DTL          219533 non-null  object
dtypes: object(8)
memory usage: 66.2+ MB


Бачу що в колонці NZ ( назви моделей телефону) є пропущені значення, для аналізу можна їх залишити , змінивши назву на "Невідомо"

In [ ]:
df['NZ'] = df['NZ'].fillna('Невідомо')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 964667 entries, 0 to 1049036
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   ID           964667 non-null  object
 1   OVD          964667 non-null  object
 2   INSERT_DATE  964667 non-null  object
 3   NZ           964667 non-null  object
 4   IMEI         964667 non-null  object
 5   NK           964537 non-null  object
 6   DK           964667 non-null  object
 7   DTL          219533 non-null  object
dtypes: object(8)
memory usage: 66.2+ MB


Прибираю колонки які не буду використовувати для подальшого аналізу. ID та IMEI мали сенс тільки для перевірки дублікатів якби були унікальними клчами, а DTL та NK взагалі не несуть корисної для аналізу інформації

In [ ]:
df=df.drop(columns=['NK','DTL','ID','IMEI'])

In [ ]:
df.head()

,OVD,INSERT_DATE,NZ,DK
0,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06T14:49:38,SAMSUNG,2020-11-05T00:00:00
1,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06T14:49:38,SAMSUNG,2020-10-23T00:00:00
2,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06T14:49:38,XIAOMI,2020-11-05T00:00:00
3,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06T14:49:39,XIAOMI,2020-11-05T00:00:00
4,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06T14:49:39,SAMSUNG,2020-10-23T00:00:00


Далі працюю з часом. Маю 2 колонки з часом INSERT_DATE(дата реєстрації звернення) та DK( дата крадіжки). У зв'язку з тим що в DK час не вказано, обидві колонки приведу до вигляду YYYY-MM-DD. Та створю колонку з різницею між датою крадіжки та датою реєстрації, вона знадобиться для подальшого аналізу

In [ ]:
df['DK'] = pd.to_datetime(df['DK'], errors='coerce')

In [ ]:
df['INSERT_DATE']=pd.to_datetime(df['INSERT_DATE'], errors='coerce')

В датасеті виявлено нерівні формати року ти типу "0210" тож треба видалити знаяення що стали NAN після преведення дати до потрібного вигляду

In [ ]:
print(df['DK'].isna().sum())
print(df['INSERT_DATE'].isna().sum())

27
0


In [ ]:
df=df.dropna(subset=['DK'])

In [ ]:
df['DK']=df['DK'].dt.normalize()
df['INSERT_DATE']=df['INSERT_DATE'].dt.normalize()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 964640 entries, 0 to 1049036
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   OVD          964640 non-null  object        
 1   INSERT_DATE  964640 non-null  datetime64[ns]
 2   NZ           964640 non-null  object        
 3   DK           964640 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 36.8+ MB


In [ ]:
df['DAYS_TO_REGISTER']=(df['INSERT_DATE']-df['DK']).dt.days

In [ ]:
df.head()

,OVD,INSERT_DATE,NZ,DK,DAYS_TO_REGISTER
0,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06,SAMSUNG,2020-11-05,1
1,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06,SAMSUNG,2020-10-23,14
2,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06,XIAOMI,2020-11-05,1
3,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06,XIAOMI,2020-11-05,1
4,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06,SAMSUNG,2020-10-23,14


Для подальшого аналізу та візуалізаціх мені потрібно щоб всі назви моделей були приведені до одного виду, тож створю нову колонку BRAND та перезапишу туди всі моделі в однаковому вигляді

In [ ]:
print(df['NZ'].nunique())
print(df['NZ'].value_counts().head(20))

80607
NZ
SAMSUNG         136495
XIAOMI           79448
APPLE IPHONE     46757
NOKIA            45297
LENOVO           30234
HUAWEI           22948
НОКИА            21100
САМСУНГ          20959
LG               15474
XІAOMІ           13979
APPLE ІPHONE     13002
MEIZU            12189
HTC              10833
FLY               9596
IPHONE            9213
HUAWEІ            8959
PRESTIGIO         6062
SONY              6034
ZTE               5823
MOTOROLA          4638
Name: count, dtype: int64


In [ ]:
def normalize_brand(name):

    name = str(name).upper().strip()

    if any(x in name for x in ['SAMSUNG', 'САМСУНГ','SAMSYNG','SAMSYNG','SUMSUNG']):
        return 'Samsung'
    if any(x in name for x in ['XIAOMI', 'REDMI', 'POCO','XІAOMІ']):
        return 'Xiaomi'
    if any(x in name for x in ['IPHONE', 'APPLE', 'АЙФОН','IPHON','IPONE']):
        return 'Apple'
    if any(x in name for x in ['NOKIA', 'НОКИА', 'НОКІА', 'НОКИЯ','NOKІA','HOKIA','НОКІЯ','NОКІА']):
        return 'Nokia'
    if any(x in name for x in ['HUAWEI', 'HONOR','HUAWEІ']):
        return 'Huawei'
    if any(x in name for x in ['LENOVO', 'ЛЕНОВО']):
        return 'Lenovo'
    if any(x in name for x in ['SONY', 'СОНИ', 'ERICSSON','SONI ERIKSON','СОНІ ЕРІКСОН','SONI']):
        return 'Sony'
    if any(x in name for x in ['MEIZU','MEІZU']):
        return 'Meizu'
    if any(x in name for x in ['LG']):
        return 'LG'
    if any(x in name for x in ['HTC', 'НТС']):
        return 'HTC'
    if any(x in name for x in ['FLY', 'ФЛАЙ']):
        return 'Fly'
    if any(x in name for x in ['ZTE']):
        return 'ZTE'
    if any(x in name for x in ['MOTOROLA','МОТОРОЛА','МОТОРОЛЛА','MOTOROLLA']):
        return 'Motorola'
    if any(x in name for x in ['PRESTIGIO','ПРЕСТИЖИО']):
        return 'Prestigio'
    if any(x in name for x in ['OPPO']):
        return 'Oppo'
    if any(x in name for x in ['VIVO']):
        return 'Vivo'
    if any(x in name for x in ['REALME']):
        return 'Realme'
    if any(x in name for x in ['ASUS']):
        return 'Asus'
    if any(x in name for x in ['ALCATEL']):
        return 'Alcatel'
    if any(x in name for x in ['TECNO', 'TECHNO']):
        return 'Tecno'
    if any(x in name for x in ['NOMI', 'НОМІ', 'НОМИ','NOMІ']):
        return 'Nomi'
    if any(x in name for x in ['BRAVIS']):
        return 'Bravis'
    if any(x in name for x in ['SIGMA']):
        return 'Sigma'
    if any(x in name for x in ['DOOGEE']):
        return 'Doogee'
    if any(x in name for x in ['BLACKVIEW']):
        return 'Blackview'
    if any(x in name for x in ['ERGO']):
        return 'Ergo'
    if any(x in name for x in ['REDMOND']):
        return 'Redmond'
    if any(x in name for x in ['СИМЕНС','SIEMENS','SIMENS']):
        return 'Siemens'

    return 'Інше'

df['BRAND'] = df['NZ'].apply(normalize_brand)
print(df['BRAND'].value_counts().head(5))

BRAND
Samsung    249143
Nokia      198015
Xiaomi      98960
Apple       88630
Lenovo      49473
Name: count, dtype: int64


In [ ]:
інше = df[df['BRAND'] == 'Інше']['NZ'].value_counts().head(5)
print(інше)

NZ
S-TELL       915
MICROSOFT    905
ONEPLUS      712
NEFF         703
GSMART       677
Name: count, dtype: int64


Тепер колонку NZ можна видалити та зберегти готовий до аналізу датасет

In [ ]:
df=df.drop(columns='NZ')

In [ ]:
df.head()

,OVD,INSERT_DATE,DK,DAYS_TO_REGISTER,BRAND
0,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06,2020-11-05,1,Samsung
1,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06,2020-10-23,14,Samsung
2,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06,2020-11-05,1,Xiaomi
3,ЖИТОМИРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЖИТОМИРСЬКІ...,2020-11-06,2020-11-05,1,Xiaomi
4,ОБОЛОНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-11-06,2020-10-23,14,Samsung


Але є проблема з розміром файлу, після очищення та підготовки даних датасет все одно налічує майже мільйон рядків, що значно ускладнює роботу з ним. Тому щоб зменшити його перевірю за які роки дані там описано та оберу період останні 5 років + 2026 рік до кінця лютого ( аналіз проводиться в березні, тож повних даних за цей місяць ще немає)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 964640 entries, 0 to 1049036
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   OVD               964640 non-null  object        
 1   INSERT_DATE       964640 non-null  datetime64[ns]
 2   DK                964640 non-null  datetime64[ns]
 3   DAYS_TO_REGISTER  964640 non-null  int64         
 4   BRAND             964640 non-null  object        
dtypes: datetime64[ns](2), int64(1), object(2)
memory usage: 44.2+ MB


In [ ]:
df['DK'].dt.year.unique()

array([2020, 2019, 2017, 2018, 2022, 2014, 2010, 2009, 2015, 2005, 2016,
       2004, 2003, 2001, 2006, 1996, 2021, 2007, 2008, 2011, 2002, 2000,
       1999, 1997, 1994, 2012, 2023, 1924, 1900, 1931, 2024, 1990, 1983,
       1991, 2025, 2026, 1998, 1995, 1910, 1980, 2013, 1921, 1984, 1987,
       1920], dtype=int32)

In [ ]:
df_five_years = df[df['DK'].dt.year.isin([2021,2022,2023,2024,2025,2026])]

Кількість рядків тепер скоротилась до 146 тисяч , що дуже добре бо значно спростить роботу з файлом, але зиявила іншу проблему, через те що не скрізь вказано коректні дати крадіжки та внесення в реєстр заяви про неї маємо в колонці з різницею кількості днів від інциденту до реєстрації мінусові значення, такого бути не може тому треба видалити ці значення

In [ ]:
df_five_years.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146241 entries, 1874 to 1038658
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   OVD               146241 non-null  object        
 1   INSERT_DATE       146241 non-null  datetime64[ns]
 2   DK                146241 non-null  datetime64[ns]
 3   DAYS_TO_REGISTER  146241 non-null  int64         
 4   BRAND             146241 non-null  object        
dtypes: datetime64[ns](2), int64(1), object(2)
memory usage: 6.7+ MB


In [ ]:
df_five_years

,OVD,INSERT_DATE,DK,DAYS_TO_REGISTER,BRAND
1874,ЧЕРКАСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРКАСЬКІЙ ОБ...,2020-06-21,2022-07-06,-745,Motorola
17851,КИЇВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ВІННИЦЬКОГО ВІДДІЛ...,2020-09-21,2021-01-25,-126,Xiaomi
38426,ХЕРСОНСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ХЕРСОНСЬКІЙ ...,2021-02-01,2021-01-28,4,Samsung
38428,ДРУЖКІВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ КРАМАТОРСЬКОГО ...,2021-02-03,2021-02-02,1,Blackview
38431,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi
...,...,...,...,...,...
344676,НОВОВОДОЛАЗЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ПЕРВОМАЙСЬКО...,2026-02-25,2026-01-23,33,Blackview
344677,НОВОВОДОЛАЗЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ПЕРВОМАЙСЬКО...,2026-02-25,2026-01-23,33,Blackview
632780,БІЛЯЇВСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ОДЕСЬКІЙ ОБЛ...,2014-05-26,2022-01-30,-2806,Xiaomi
1038657,ДЕСНЯНСЬКЕ УПРАВЛІННЯ ПОЛІЦІЇ ГУНП В М. КИЄВІ,2020-03-17,2021-05-14,-423,Xiaomi


In [ ]:
print((df_five_years['DAYS_TO_REGISTER'] < 0).sum())

388


In [ ]:
df_five_years = df_five_years[df_five_years['DAYS_TO_REGISTER'] >= 0]

In [ ]:
print((df_five_years['DAYS_TO_REGISTER'] < 0).sum())

0


In [ ]:
df_five_years

,OVD,INSERT_DATE,DK,DAYS_TO_REGISTER,BRAND
38426,ХЕРСОНСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ХЕРСОНСЬКІЙ ...,2021-02-01,2021-01-28,4,Samsung
38428,ДРУЖКІВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ КРАМАТОРСЬКОГО ...,2021-02-03,2021-02-02,1,Blackview
38431,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi
38432,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi
38433,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi
...,...,...,...,...,...
344671,АМУР-НИЖНЬОДНІПРОВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ДНІП...,2026-02-11,2026-02-11,0,Huawei
344672,АМУР-НИЖНЬОДНІПРОВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ДНІП...,2026-02-11,2026-02-11,0,Huawei
344675,ОБУХІВСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В КИЇВСЬКІЙ ОБ...,2026-02-12,2026-02-05,7,Xiaomi
344676,НОВОВОДОЛАЗЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ПЕРВОМАЙСЬКО...,2026-02-25,2026-01-23,33,Blackview


Зберігаю відфільтрований датасет

In [ ]:
df_five_years['OVD'] = df_five_years['OVD'].str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False)
df_five_years['BRAND'] = df_five_years['BRAND'].str.replace('\n', ' ', regex=False).str.replace('\r', ' ', regex=False)

df_five_years.to_csv('stolen_phones_5y.csv', index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL)

In [ ]:
files.download('stolen_phones_5y.csv')

Для візуалізації співвідношення частки ринку мобільних постачальників України та частки вкрадених телефонів по топ 3 брендах побудую таблицю по роках


In [ ]:
df_five_years

,OVD,INSERT_DATE,DK,DAYS_TO_REGISTER,BRAND
38426,ХЕРСОНСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ХЕРСОНСЬКІЙ ...,2021-02-01,2021-01-28,4,Samsung
38428,ДРУЖКІВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ КРАМАТОРСЬКОГО ...,2021-02-03,2021-02-02,1,Blackview
38431,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi
38432,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi
38433,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi
...,...,...,...,...,...
344671,АМУР-НИЖНЬОДНІПРОВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ДНІП...,2026-02-11,2026-02-11,0,Huawei
344672,АМУР-НИЖНЬОДНІПРОВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ДНІП...,2026-02-11,2026-02-11,0,Huawei
344675,ОБУХІВСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В КИЇВСЬКІЙ ОБ...,2026-02-12,2026-02-05,7,Xiaomi
344676,НОВОВОДОЛАЗЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ПЕРВОМАЙСЬКО...,2026-02-25,2026-01-23,33,Blackview


In [ ]:
df_five_years['YEAR']=df_five_years['DK'].dt.year

/tmp/ipykernel_202/1076010343.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_five_years['YEAR']=df_five_years['DK'].dt.year


In [ ]:
df_five_years

,OVD,INSERT_DATE,DK,DAYS_TO_REGISTER,BRAND,YEAR
38426,ХЕРСОНСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ХЕРСОНСЬКІЙ ...,2021-02-01,2021-01-28,4,Samsung,2021
38428,ДРУЖКІВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ КРАМАТОРСЬКОГО ...,2021-02-03,2021-02-02,1,Blackview,2021
38431,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi,2021
38432,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi,2021
38433,НОВГОРОД-СІВЕРСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В ЧЕРН...,2021-02-03,2021-01-27,7,Xiaomi,2021
...,...,...,...,...,...,...
344671,АМУР-НИЖНЬОДНІПРОВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ДНІП...,2026-02-11,2026-02-11,0,Huawei,2026
344672,АМУР-НИЖНЬОДНІПРОВСЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ДНІП...,2026-02-11,2026-02-11,0,Huawei,2026
344675,ОБУХІВСЬКИЙ ВІДДІЛ ПОЛІЦІЇ ГУНП В КИЇВСЬКІЙ ОБ...,2026-02-12,2026-02-05,7,Xiaomi,2026
344676,НОВОВОДОЛАЗЬКЕ ВІДДІЛЕННЯ ПОЛІЦІЇ ПЕРВОМАЙСЬКО...,2026-02-25,2026-01-23,33,Blackview,2026


Всі наступні поля для додаткових розрахунків , потреба в яких виникла в ході подальшого аналізу та візуалізації даних.

In [ ]:
df_total=df_five_years.groupby(['YEAR'])['BRAND'].count().reset_index()

In [ ]:
df_total

,YEAR,BRAND
0,2021,53535
1,2022,27236
2,2023,29121
3,2024,19558
4,2025,14457
5,2026,1946


In [ ]:
df_braand=df_five_years.groupby(['YEAR','BRAND'])['OVD'].count().reset_index()

In [ ]:
df_braand

,YEAR,BRAND,OVD
0,2021,Alcatel,42
1,2021,Apple,11069
2,2021,Asus,70
3,2021,Blackview,169
4,2021,Bravis,49
...,...,...,...
163,2026,Tecno,38
164,2026,Vivo,8
165,2026,Xiaomi,602
166,2026,ZTE,57


In [ ]:
mask=df_braand['BRAND'].isin(['Samsung','Apple','Xiaomi'])

In [ ]:
df_braand=df_braand[mask]

In [ ]:
df_braand

,YEAR,BRAND,OVD
1,2021,Apple,11069
20,2021,Samsung,11801
26,2021,Xiaomi,17089
30,2022,Apple,5164
49,2022,Samsung,6274
55,2022,Xiaomi,8763
59,2023,Apple,3669
78,2023,Samsung,6691
84,2023,Xiaomi,10717
88,2024,Apple,2838


In [ ]:
df_braand=df_braand.merge( df_total, on='YEAR', how='inner')

In [ ]:
df_braand

,YEAR,BRAND_x,OVD,BRAND_y
0,2021,Apple,11069,53535
1,2021,Samsung,11801,53535
2,2021,Xiaomi,17089,53535
3,2022,Apple,5164,27236
4,2022,Samsung,6274,27236
5,2022,Xiaomi,8763,27236
6,2023,Apple,3669,29121
7,2023,Samsung,6691,29121
8,2023,Xiaomi,10717,29121
9,2024,Apple,2838,19558


In [ ]:
df_braand['SHARE']=df_braand['OVD']/df_braand['BRAND_y']

In [ ]:
df_braand=df_braand.rename(columns={'BRAND_x':'BRAND','OVD':'COUNT','BRAND_y':'TOTAL_COUNT'})

In [ ]:
df_braand

,YEAR,BRAND,COUNT,TOTAL_COUNT,SHARE
0,2021,Apple,11069,53535,0.206762
1,2021,Samsung,11801,53535,0.220435
2,2021,Xiaomi,17089,53535,0.319212
3,2022,Apple,5164,27236,0.189602
4,2022,Samsung,6274,27236,0.230357
5,2022,Xiaomi,8763,27236,0.321743
6,2023,Apple,3669,29121,0.125992
7,2023,Samsung,6691,29121,0.229765
8,2023,Xiaomi,10717,29121,0.368016
9,2024,Apple,2838,19558,0.145107


In [ ]:
import csv
df_braand.to_csv('Share stolen brand.csv', index=False, encoding='utf-8-sig', quoting=csv.QUOTE_ALL)

In [ ]:
files.download('Share stolen brand.csv')

In [ ]:
df_try=df_five_years['DAYS_TO_REGISTER'].isin(range(1,31))

In [ ]:
df_try.info()

<class 'pandas.core.series.Series'>
Index: 145853 entries, 38426 to 344677
Series name: DAYS_TO_REGISTER
Non-Null Count   Dtype
--------------   -----
145853 non-null  bool 
dtypes: bool(1)
memory usage: 1.3 MB
